In [2]:
premium = 400.0
claims = 300.0
loss_ratio = claims / premium
print(f"Loss ratio: {loss_ratio:.1%}")
if loss_ratio > 1 : print("Unprofitable")
else: print("Profitable")

Loss ratio: 75.0%
Profitable


In [3]:
import duckdb

con = duckdb.connect()
con.sql("SELECT 1 + 1 AS test").show()

┌───────┐
│ test  │
│ int32 │
├───────┤
│     2 │
└───────┘



In [4]:
con.sql("""Select * From read_csv('data/motor_data.csv', delim=';', nullstr='NA')Limit 5""").show()

┌───────┬─────────────────────┬───────────────────┬───────────────────┬────────────┬──────────────────────┬──────────────────────┬───────────┬───────────────────┬──────────────┬──────────────┬───────┬────────────┬─────────┬─────────┬──────────────────┬───────────────┬──────────────────┬──────────────────┬───────────┬───────┬───────────────┬────────────────────┬───────┬───────────────────┬───────────────┬─────────┬───────────┬────────┬────────┐
│  ID   │ Date_start_contract │ Date_last_renewal │ Date_next_renewal │ Date_birth │ Date_driving_licence │ Distribution_channel │ Seniority │ Policies_in_force │ Max_policies │ Max_products │ Lapse │ Date_lapse │ Payment │ Premium │ Cost_claims_year │ N_claims_year │ N_claims_history │ R_Claims_history │ Type_risk │ Area  │ Second_driver │ Year_matriculation │ Power │ Cylinder_capacity │ Value_vehicle │ N_doors │ Type_fuel │ Length │ Weight │
│ int64 │        date         │       date        │       date        │    date    │         date       

In [5]:
con.sql("""Create or Replace View raw as Select * From read_csv('data/motor_data.csv', delim=';', nullstr='NA')""").show()

┌────────┐
│ Count  │
│ int64  │
└────────┘
  0 rows



In [6]:
con.sql("""Select Count(*) as n_rows, Count(Distinct ID) as n_policies From raw """).show()

┌────────┬────────────┐
│ n_rows │ n_policies │
│ int64  │   int64    │
├────────┼────────────┤
│ 105555 │      53502 │
└────────┴────────────┘



In [7]:
con.sql("""select id, count(*) as n_years from raw group by id order by id limit 10 """).show()

┌───────┬─────────┐
│  ID   │ n_years │
│ int64 │  int64  │
├───────┼─────────┤
│     1 │       4 │
│     2 │       2 │
│     3 │       4 │
│     4 │       3 │
│     5 │       2 │
│     6 │       3 │
│     7 │       1 │
│     8 │       3 │
│     9 │       2 │
│    10 │       2 │
└───────┴─────────┘
      10 rows    



In [8]:
con.sql("""select n_years , count(*) as n_policies from (select id, count(*) as n_years from raw group by id ) group by n_years order by n_years""").show()

┌─────────┬────────────┐
│ n_years │ n_policies │
│  int64  │   int64    │
├─────────┼────────────┤
│       1 │      20579 │
│       2 │      14685 │
│       3 │      17346 │
│       4 │        892 │
└─────────┴────────────┘



In [9]:
con.sql("""select sum(premium) as total_premium, sum(cost_claims_year) as total_claims, sum(cost_claims_year)/sum(premium) as loss_ratio, 
sum(case when n_claims_year = 0 then 1 else 0 end) as zero_claims_rows from raw """).show()

┌───────────────────┬────────────────────┬────────────────────┬──────────────────┐
│   total_premium   │    total_claims    │     loss_ratio     │ zero_claims_rows │
│      double       │       double       │       double       │      int128      │
├───────────────────┼────────────────────┼────────────────────┼──────────────────┤
│ 33344038.87999989 │ 16208741.370000124 │ 0.4861061201473796 │            85909 │
└───────────────────┴────────────────────┴────────────────────┴──────────────────┘



In [10]:
con.sql("""select type_risk, count(*) as n_rows, Round(sum(premium),0) as total_premium, Round(sum(cost_claims_year),0) as total_claims, 
Round(sum(cost_claims_year)/sum(premium),3) as loss_ratio
 from raw group by type_risk order by loss_ratio desc """).show()

┌───────────┬────────┬───────────────┬──────────────┬────────────┐
│ Type_risk │ n_rows │ total_premium │ total_claims │ loss_ratio │
│   int64   │ int64  │    double     │    double    │   double   │
├───────────┼────────┼───────────────┼──────────────┼────────────┤
│         2 │  13212 │     4337095.0 │    2197753.0 │      0.507 │
│         3 │  82990 │    27716103.0 │   13701739.0 │      0.494 │
│         1 │   8502 │     1223142.0 │     309056.0 │      0.253 │
│         4 │    851 │       67698.0 │        193.0 │      0.003 │
└───────────┴────────┴───────────────┴──────────────┴────────────┘



In [11]:
con.sql("""select type_risk, count(*) as n_rows, sum(n_claims_year) as n_claims, Round(sum(n_claims_year)/count(*), 3) as frequency, 
Round(sum(cost_claims_year)/sum(n_claims_year),0) as severity, Round(sum(premium)/count(*), 3) as avg_premium, Round(sum(cost_claims_year)/sum(premium),3) as loss_ratio
 from raw group by type_risk order by type_risk """).show()

┌───────────┬────────┬──────────┬───────────┬──────────┬─────────────┬────────────┐
│ Type_risk │ n_rows │ n_claims │ frequency │ severity │ avg_premium │ loss_ratio │
│   int64   │ int64  │  int128  │  double   │  double  │   double    │   double   │
├───────────┼────────┼──────────┼───────────┼──────────┼─────────────┼────────────┤
│         1 │   8502 │     1236 │     0.145 │    250.0 │     143.865 │      0.253 │
│         2 │  13212 │     7499 │     0.568 │    293.0 │     328.269 │      0.507 │
│         3 │  82990 │    32926 │     0.397 │    416.0 │     333.969 │      0.494 │
│         4 │    851 │        1 │     0.001 │    193.0 │      79.552 │      0.003 │
└───────────┴────────┴──────────┴───────────┴──────────┴─────────────┴────────────┘



In [12]:
con.sql("""
    select id, date_birth, date_last_renewal,
           date_diff('year', date_birth, date_last_renewal) as age_diff,
           date_sub('year', date_birth, date_last_renewal) as age_sub
    from raw
    where date_diff('year', date_birth, date_last_renewal)
       <> date_sub('year', date_birth, date_last_renewal)
    limit 5
""").show()

┌───────┬────────────┬───────────────────┬──────────┬─────────┐
│  ID   │ Date_birth │ Date_last_renewal │ age_diff │ age_sub │
│ int64 │    date    │       date        │  int64   │  int64  │
├───────┼────────────┼───────────────────┼──────────┼─────────┤
│     5 │ 1973-07-10 │ 2017-05-12        │       44 │      43 │
│     5 │ 1973-07-10 │ 2018-05-12        │       45 │      44 │
│     6 │ 1966-10-21 │ 2016-05-29        │       50 │      49 │
│     6 │ 1966-10-21 │ 2017-05-29        │       51 │      50 │
│     6 │ 1966-10-21 │ 2018-05-29        │       52 │      51 │
└───────┴────────────┴───────────────────┴──────────┴─────────┘



In [13]:
con.sql("""
    select count(*) as diff_not_sub
    from raw
    where (date_diff('year', date_birth, date_last_renewal) - date_sub('year', date_birth, date_last_renewal)) <> 0
    
""").show()

┌──────────────┐
│ diff_not_sub │
│    int64     │
├──────────────┤
│        50057 │
└──────────────┘



In [18]:
con.sql("""
    create or replace view policy_year as
    select *,
           date_sub('year', date_birth, date_last_renewal) as driver_age,
           case when date_sub('year', date_driving_licence, date_last_renewal) < 0
                then null
                else date_sub('year', date_driving_licence, date_last_renewal)
           end as years_licensed
    from raw
""")

CountBIGINT


In [15]:
con.sql("""select min(driver_age) as minDA, max(driver_age) as maxDA, avg(driver_age) as avgDA, 
min(years_licensed) as minYL, max(years_licensed) as maxYL, avg(years_licensed) as avgYL, from policy_year """).show()

┌───────┬───────┬───────────────────┬───────┬───────┬───────────────────┐
│ minDA │ maxDA │       avgDA       │ minYL │ maxYL │       avgYL       │
│ int64 │ int64 │      double       │ int64 │ int64 │      double       │
├───────┼───────┼───────────────────┼───────┼───────┼───────────────────┤
│    18 │    98 │ 46.82727488039411 │    -2 │    74 │ 24.27833830704372 │
└───────┴───────┴───────────────────┴───────┴───────┴───────────────────┘



In [22]:
con.sql("""select case when years_licensed is null then 'missing' else 'ok' end as licence_flag, count(*) as n_rows,
Round(sum(cost_claims_year)/sum(premium),3) as loss_ratio  from policy_year group by licence_flag """).show()

┌──────────────┬────────┬────────────┐
│ licence_flag │ n_rows │ loss_ratio │
│   varchar    │ int64  │   double   │
├──────────────┼────────┼────────────┤
│ ok           │ 105550 │      0.486 │
│ missing      │      5 │        0.0 │
└──────────────┴────────┴────────────┘

